In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError("Invalid catalog name.")

spark.conf.set("spark.sql.session.timeZone", "UTC")

customer_source_table = f"{catalog}.silver.customers_current"
customer_history_table = f"{catalog}.silver.customers_history"

In [ ]:
customer_profiles = (
    spark.table(customer_source_table)
    .select(
        "customer_id",
        "customer_name",
        "email",
        "country_code",
        "customer_segment",
        "is_active",
        F.col("created_at").alias("source_created_at"),
        F.col("updated_at").alias("source_updated_at")
    )
)

display(
    customer_profiles.orderBy("customer_id")
)

In [ ]:
invalid_customer_ids = customer_profiles.filter(
    F.col("customer_id").isNull()
    | (F.length(F.trim(F.col("customer_id"))) == 0)
)

if invalid_customer_ids.limit(1).count() > 0:
    display(invalid_customer_ids)
    raise ValueError("Customer profiles contain missing IDs.")

duplicate_customer_ids = (
    customer_profiles
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_customer_ids.limit(1).count() > 0:
    display(duplicate_customer_ids)
    raise ValueError(
        "Customer history requires one current profile per customer."
    )

print("Customer history source checks passed.")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {customer_history_table} (
        customer_id STRING,
        customer_name STRING,
        email STRING,
        country_code STRING,
        customer_segment STRING,
        is_active BOOLEAN,
        source_created_at TIMESTAMP,
        source_updated_at TIMESTAMP,
        valid_from TIMESTAMP,
        valid_to TIMESTAMP,
        is_current BOOLEAN
    )
    USING DELTA
""")

In [ ]:
from datetime import datetime, timezone
from functools import reduce
from delta.tables import DeltaTable


tracked_attributes = [
    "customer_name",
    "email",
    "country_code",
    "customer_segment",
    "is_active"
]

history_current = (
    spark.table(customer_history_table)
    .filter(F.col("is_current"))
)

# Stop if the existing history is already inconsistent.
duplicate_current_versions = (
    history_current
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_current_versions.limit(1).count() > 0:
    display(duplicate_current_versions)
    raise ValueError(
        "Customer history contains multiple current versions."
    )

# True only when every tracked attribute matches.
attributes_unchanged = reduce(
    lambda left, right: left & right,
    [
        F.col(f"source.{column_name}").eqNullSafe(
            F.col(f"history.{column_name}")
        )
        for column_name in tracked_attributes
    ]
)

changed_profiles = (
    customer_profiles.alias("source")
    .join(
        history_current.alias("history"),
        F.col("source.customer_id") == F.col("history.customer_id"),
        how="inner"
    )
    .filter(~attributes_unchanged)
    .select("source.*")
)

display(changed_profiles)

In [ ]:
# One fixed observation time for this operation.
observation_time = datetime.now(timezone.utc)

# Every source profile can match its existing current history row.
profiles_to_match = customer_profiles.withColumn(
    "merge_customer_id",
    F.col("customer_id")
)

# Changed profiles also need a new row.
# A null merge key deliberately prevents these rows from matching.
profiles_to_insert = changed_profiles.withColumn(
    "merge_customer_id",
    F.lit(None).cast("string")
)

staged_profiles = (
    profiles_to_match
    .unionByName(profiles_to_insert)
    .withColumn(
        "observed_at",
        F.lit(observation_time).cast("timestamp")
    )
)

merge_attributes_unchanged = reduce(
    lambda left, right: left & right,
    [
        F.col(f"target.{column_name}").eqNullSafe(
            F.col(f"source.{column_name}")
        )
        for column_name in tracked_attributes
    ]
)

insert_values = {
    column_name: f"source.{column_name}"
    for column_name in customer_profiles.columns
}

insert_values.update({
    "valid_from": "source.observed_at",
    "valid_to": "CAST(NULL AS TIMESTAMP)",
    "is_current": "true"
})

(
    DeltaTable.forName(spark, customer_history_table)
    .alias("target")
    .merge(
        staged_profiles.alias("source"),
        """
        target.customer_id = source.merge_customer_id
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition=~merge_attributes_unchanged,
        set={
            "valid_to": "source.observed_at",
            "is_current": "false"
        }
    )
    .whenNotMatchedInsert(
        values=insert_values
    )
    .execute()
)

print("Customer history merge completed.")

In [ ]:
saved_customer_history = spark.table(
    customer_history_table
)

display(
    saved_customer_history.orderBy(
        "customer_id",
        "valid_from"
    )
)

display(
    saved_customer_history
    .groupBy("is_current")
    .count()
)

In [ ]:
from pyspark.sql.window import Window


history = spark.table(customer_history_table)

current_history = history.filter(
    F.col("is_current")
)

# 1. Each customer must have exactly one current version.
current_version_counts = (
    history
    .groupBy("customer_id")
    .agg(
        F.sum(
            F.when(F.col("is_current"), 1).otherwise(0)
        ).alias("current_version_count")
    )
)

invalid_current_counts = current_version_counts.filter(
    F.col("current_version_count") != 1
)

if invalid_current_counts.limit(1).count() > 0:
    display(invalid_current_counts)
    raise ValueError(
        "Each customer in history must have exactly one current version."
    )


# 2. Current business attributes must match the source.
comparison_columns = ["customer_id"] + tracked_attributes

expected_profiles = customer_profiles.select(
    *comparison_columns
)

actual_profiles = current_history.select(
    *comparison_columns
)

if (
    expected_profiles.exceptAll(actual_profiles).limit(1).count() > 0
    or actual_profiles.exceptAll(expected_profiles).limit(1).count() > 0
):
    raise ValueError(
        "Current customer history does not match the source profiles."
    )


# 3. Check the validity of each version's interval.
invalid_intervals = history.filter(
    F.col("valid_from").isNull()
    | F.col("is_current").isNull()
    | (
        F.col("is_current")
        & F.col("valid_to").isNotNull()
    )
    | (
        (~F.col("is_current"))
        & (
            F.col("valid_to").isNull()
            | (F.col("valid_to") <= F.col("valid_from"))
        )
    )
)

if invalid_intervals.limit(1).count() > 0:
    display(invalid_intervals)
    raise ValueError("Customer history contains invalid intervals.")


# 4. A customer must not have two versions with the same start time.
duplicate_version_starts = (
    history
    .groupBy("customer_id", "valid_from")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_version_starts.limit(1).count() > 0:
    display(duplicate_version_starts)
    raise ValueError("Duplicate customer history version start times.")


# 5. Each old version must end exactly when the next version starts.
timeline_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("valid_from")
)

history_timeline = history.withColumn(
    "next_valid_from",
    F.lead("valid_from").over(timeline_window)
)

invalid_timelines = history_timeline.filter(
    (
        F.col("next_valid_from").isNotNull()
        & (
            F.col("is_current")
            | ~F.col("valid_to").eqNullSafe(
                F.col("next_valid_from")
            )
        )
    )
    | (
        F.col("next_valid_from").isNull()
        & (~F.col("is_current"))
    )
)

if invalid_timelines.limit(1).count() > 0:
    display(invalid_timelines)
    raise ValueError(
        "Customer history contains gaps, overlaps, "
        "or an incorrectly marked latest version."
    )

print("Customer history verification passed.")